# Dataset Preparation for ML

In [1]:
%load_ext autoreload
%autoreload 2

import warnings
from IPython.display import display
import pandas as pd

warnings.filterwarnings("ignore", message="IProgress not found.*")

from src.tensor_utils import (
    build_dataset_tensor_embedding_2d,
    build_moa_labeled_tensor_dataset,
    save_labeled_tensor_dataset,
    plot_tensor_embedding_2d,
)

from src.notebook_utils import (
    configure_full_dataframe_display,
    load_compound_image_condition_map_csv,
)


In [2]:
configure_full_dataframe_display()

# Load the previously generated condition map snapshot instead of rebuilding it from the workbook and folders.
condition_df = load_compound_image_condition_map_csv()
condition_df.shape

(2376, 20)

In [3]:
# User inputs
# selected_mechanisms:
#   list of normalized mechanism names from condition_df["mechanism_of_action"].
#   Use 2 or more classes depending on the comparison you want to study.
#   This default picks two similar pairs:
#   Pair 1: GABAAR_Antagonist vs GABAAR_NegativeAllostericModulator.
#   Pair 2: NMDAR_Activation vs NMDAR_Antagonist.
#   The goal is to create within-pair similarity but between-pair separation.
selected_mechanisms = [
    "GABAAR_Antagonist",
    "GABAAR_NegativeAllostericModulator",
    "NMDAR_Activation",
    "NMDAR_Antagonist",
]
# selected_concentrations:
#   subset of treatment concentration bands to include.
#   Typical options: ["high"], ["mid"], ["low"], or combinations such as ["high", "mid", "low"].
#   Water controls are added separately as class 0 and do not need to be listed here.
selected_concentrations = ["high", "mid"]
# max_compounds_per_action:
#   maximum number of distinct compounds to keep for each mechanism class.
#   Use a small number for quick prototyping and a larger number for broader class coverage.
max_compounds_per_action = 2
# max_tensors_per_compound:
#   maximum number of condition tensors to keep per compound after filtering.
#   Increase this to use more runs / folders per compound, decrease it to balance classes faster.
max_tensors_per_compound = 32
# output_size:
#   target tensor size as (T, Z, Y, X) after cached loading, downsampling, and drift correction.
#   Use None in any position to keep the source dimension unchanged.
#   Examples: (10, 1, 32, 32), (10, 3, 64, 64), (None, 1, 64, 64).
output_size = (30, 5, 96, 96)
# only_active:
#   True keeps only folders not marked do_not_use / inactive.
#   False also includes non-active folders when present in the condition map.
only_active = True
# normalize_global_drift:
#   True applies the LOESS-style global intensity drift correction during loading.
#   False keeps the raw intensity trend.
normalize_global_drift = True
# loess_frac:
#   smoothing fraction for the global intensity drift correction.
#   Smaller values follow local changes more closely; larger values smooth more aggressively.
#   Typical range: 0.1 to 0.4.
loess_frac = 0.25
# use_cache:
#   True reuses cached processed tensors from .tensor_cache when available.
#   False forces a rebuild from the source TIFFs.
use_cache = True
# use_tiff_cache:
#   True mirrors only the selected TIFF files into .tiff_cache before loading.
#   False reads directly from the mounted source paths.
use_tiff_cache = True
# embedding_method:
#   "pca" for a linear 2D projection, "umap" for a nonlinear neighborhood-preserving embedding.
embedding_method = "umap"
# umap_n_neighbors:
#   local neighborhood size for UMAP. Smaller values emphasize fine local structure,
#   larger values emphasize broader global structure.
umap_n_neighbors = 32
# umap_min_dist:
#   minimum spacing of points in the UMAP embedding.
#   Smaller values make tighter clusters, larger values spread clusters out more.
umap_min_dist = 0.1
# embedding_target:
#   target used for coloring and labeling the 2D embedding.
#   Use one of "mechanism", "compound", "concentration", or "control".
embedding_target = "mechanism"
# embedding_random_seed:
#   seed used for reproducible 2D embedding initialization.
embedding_random_seed = 0
# dataset_artifact_path:
#   relative path under .dataset_cache or an absolute path for the persisted dataset artifact.
#   By default this is constructed from the active dataset settings.
output_size_tag = "_".join(
    f"{axis.lower()}{size if size is not None else 'full'}"
    for axis, size in zip(("T", "Z", "Y", "X"), output_size)
)
dataset_artifact_path = (
    f"moa_{len(selected_mechanisms)}class_"
    f"c{len(selected_concentrations)}_"
    f"mca{max_compounds_per_action}_"
    f"mtc{max_tensors_per_compound}_"
    f"{output_size_tag}.pt"
)
dataset_artifact_path

'moa_4class_c2_mca2_mtc32_t30_z5_y96_x96.pt'

In [ ]:
%%time
dataset = build_moa_labeled_tensor_dataset(
    condition_df=condition_df,
    selected_mechanisms=selected_mechanisms,
    selected_concentrations=selected_concentrations,
    max_compounds_per_action=max_compounds_per_action,
    max_tensors_per_compound=max_tensors_per_compound,
    output_size=output_size,
    only_active=only_active,
    normalize_global_drift=normalize_global_drift,
    loess_frac=loess_frac,
    use_cache=use_cache,
    use_tiff_cache=use_tiff_cache,
)

[001/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=00:01
[002/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=48:38:24


/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/src/tensor_utils.py:662: UserWarning: Failed to build treatment tensor dataset example for mechanism='GABAAR_Antagonist', compound='Bemegride', concentration_band='high', image_condition_dir='/mnt/tyler/Matt Winter/BRAIN IMAGES BACKUP/THIRD BATCH convulsants/Bemegride run 1 1 June 17/1mM F2'. Skipping this example. Root cause: OSError(112, 'Host is down')
  warnings.warn(f"{message}. Skipping this example. Root cause: {exc!r}")


[003/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=30:14:01


/media/fabrizio/06bb7271-2161-43a4-91f1-98f9b67e9ab2/home/fabrizio/code/ZebraFish/src/tensor_utils.py:662: UserWarning: Failed to build treatment tensor dataset example for mechanism='GABAAR_Antagonist', compound='Bemegride', concentration_band='high', image_condition_dir='/mnt/tyler/Matt Winter/BRAIN IMAGES BACKUP/THIRD BATCH convulsants/Bemegride run 1 1 June 17/1mM F3'. Skipping this example. Root cause: FileNotFoundError('No TIFF files found under /mnt/tyler/Matt Winter/BRAIN IMAGES BACKUP/THIRD BATCH convulsants/Bemegride run 1 1 June 17/1mM F3')
  warnings.warn(f"{message}. Skipping this example. Root cause: {exc!r}")


[004/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=21:18:05
[005/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=16:03:11
[006/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=12:53:43
[007/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=10:47:28
[008/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=high eta=9:16:56
[009/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=mid eta=8:09:29
[010/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=mid eta=7:16:59
[011/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=mid eta=6:34:45
[012/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride concentration=mid eta=6:00:19
[013/165] kind=treatment mechanism=GABAAR_Antagonist compound=Bemegride c

<tifffile.TiffPages @16> invalid page offset 7002616
<tifffile.TiffPages @16> invalid page offset 7002616
<tifffile.TiffPages @16> invalid page offset 7002616
<tifffile.TiffPages @16> invalid page offset 7002616


[082/165] kind=treatment mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=mid eta=45:05
[083/165] kind=treatment mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=mid eta=44:16
[084/165] kind=treatment mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=mid eta=43:34
[085/165] kind=control mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=control eta=42:47
[086/165] kind=control mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=control eta=42:01
[087/165] kind=control mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=control eta=41:15
[088/165] kind=control mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=control eta=40:31
[089/165] kind=control mechanism=NMDAR_Activation compound=(RS)-(Tetrazol-5-yl)glycine concentration=control eta=39:48
[090/165] kind=control mechanism=NMDAR_Activation comp

In [ ]:
saved_dataset_path = save_labeled_tensor_dataset(dataset, dataset_artifact_path)
saved_dataset_path

In [ ]:
summary_df = pd.DataFrame(
    [
        {"metric": "n_examples", "value": int(dataset["tensors"].shape[0])},
        {"metric": "tensor_shape", "value": str(tuple(dataset["tensors"].shape))},
        {"metric": "mechanism_label_tensor_shape", "value": str(tuple(dataset["labels"].shape))},
        {"metric": "compound_label_tensor_shape", "value": str(tuple(dataset["compound_labels"].shape))},
        {"metric": "concentration_label_tensor_shape", "value": str(tuple(dataset["concentration_labels"].shape))},
        {"metric": "is_control_tensor_shape", "value": str(tuple(dataset["is_control"].shape))},
        {"metric": "timepoints_per_tensor", "value": int(dataset["tensors"].shape[1])},
        {"metric": "z_slices_per_tensor", "value": int(dataset["tensors"].shape[2])},
        {"metric": "image_height", "value": int(dataset["tensors"].shape[3])},
        {"metric": "image_width", "value": int(dataset["tensors"].shape[4])},
        {"metric": "n_mechanism_classes_including_water", "value": int(len(dataset["label_map"]))},
        {"metric": "n_compound_classes_including_control", "value": int(len(dataset["compound_label_map"]))},
        {"metric": "n_concentration_classes_including_control", "value": int(len(dataset["concentration_label_map"]))},
    ]
)

mechanism_label_map_df = pd.DataFrame(
    [{"label": label, "class_name": class_name} for label, class_name in dataset["label_map"].items()]
).sort_values("label").reset_index(drop=True)
compound_label_map_df = pd.DataFrame(
    [{"label": label, "compound_name": class_name} for label, class_name in dataset["compound_label_map"].items()]
).sort_values("label").reset_index(drop=True)
concentration_label_map_df = pd.DataFrame(
    [{"label": label, "concentration_name": class_name} for label, class_name in dataset["concentration_label_map"].items()]
).sort_values("label").reset_index(drop=True)

display(summary_df)
display(mechanism_label_map_df)
display(compound_label_map_df)
display(concentration_label_map_df)

In [ ]:
display(dataset["metadata"].head(5))
display(dataset["metadata"].groupby(["label", "label_name"]).size().reset_index(name="n_examples"))
display(dataset["metadata"].groupby(["compound_label", "compound_label_name"]).size().reset_index(name="n_examples"))
display(dataset["metadata"].groupby(["concentration_label_id", "concentration_label_name"]).size().reset_index(name="n_examples"))

In [ ]:
# Reduce the persisted dataset tensors to 2D for quick inspection by the chosen target.
embedding_df = build_dataset_tensor_embedding_2d(
    dataset,
    target=embedding_target,
    method=embedding_method,
    random_state=embedding_random_seed,
    umap_n_neighbors=umap_n_neighbors,
    umap_min_dist=umap_min_dist,
)
display(embedding_df.head(5))

In [ ]:
plot_tensor_embedding_2d(
    embedding_df,
    title=f"{embedding_method.upper()} tensor embedding | target={embedding_target}",
    show_svm_background=True,
);